### Datarisk Case - Data Science

Este notebook é o principal entregável do case e vai concentrar todo o desenvolvimento da solução. As etapas planejadas são:

**Etapa 1 - Exploração individual das bases**
Analisar cada uma das 4 bases (cadastral, info, pagamentos_desenvolvimento e pagamentos_teste) separadamente: estrutura, tipos, nulos, duplicatas e distribuições, antes de qualquer cruzamento entre elas.

**Etapa 2 - Merge e construção da base-mãe**
Unir as bases via ID_CLIENTE e SAFRA_REF, construindo a base consolidada que será usada na modelagem. É nessa etapa também que construiremos a variável target (atraso >= 5 dias).

**Etapa 3 - Feature engineering**
Criar novas variáveis, tratar componentes existentes (datas, categóricas, valores faltantes) e preparar a base final pronta pra alimentar os modelos.

**Etapa 4 - Modelagem**
Treinar e comparar modelos de machine learning para estimar a probabilidade de inadimplência.

**Etapa 5 - Interpretação dos resultados**
Avaliar performance do modelo, entender as variáveis mais relevantes e validar se as previsões fazem sentido de negócio.

Mais detalhes acerca do contexto do projeto, decisões tomadas e conclusões podem ser encontradas no notebook "relatorio_datarisk.ipynb".

In [11]:
import pandas as pd



In [19]:
# Carregando as 4 bases fornecidas para o case e ajustando o separador para ';'
cadastral = pd.read_csv('../data/base_cadastral.csv', sep=';')
info = pd.read_csv('../data/base_info.csv', sep=';')
pagamentos_dev = pd.read_csv('../data/base_pagamentos_desenvolvimento.csv', sep=';')
pagamentos_teste = pd.read_csv('../data/base_pagamentos_teste.csv', sep=';')


#### Base Cadastral

In [ ]:
# Visão geral da estrutura
cadastral.head()

,ID_CLIENTE,DATA_CADASTRO,DDD,FLAG_PF,SEGMENTO_INDUSTRIAL,DOMINIO_EMAIL,PORTE,CEP_2_DIG
0,1661240395903230676,2013-08-22,99,NaN,Serviços,YAHOO,PEQUENO,65
1,8274986328479596038,2017-01-25,31,NaN,Comércio,YAHOO,MEDIO,77
2,345447888460137901,2000-08-15,75,NaN,Serviços,HOTMAIL,PEQUENO,48
3,1003144834589372198,2017-08-06,49,NaN,Serviços,OUTLOOK,PEQUENO,89
4,324916756972236008,2011-02-14,88,NaN,Serviços,GMAIL,GRANDE,62


In [21]:
cadastral.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   ID_CLIENTE           1315 non-null   int64 
 1   DATA_CADASTRO        1315 non-null   object
 2   DDD                  1078 non-null   object
 3   FLAG_PF              66 non-null     object
 4   SEGMENTO_INDUSTRIAL  1232 non-null   object
 5   DOMINIO_EMAIL        1285 non-null   object
 6   PORTE                1274 non-null   object
 7   CEP_2_DIG            1315 non-null   object
dtypes: int64(1), object(7)
memory usage: 82.3+ KB


In [22]:
cadastral['FLAG_PF'].value_counts()

FLAG_PF
X    66
Name: count, dtype: int64

In [23]:
# Convertendo DATA_CADASTRO para datetime
cadastral['DATA_CADASTRO'] = pd.to_datetime(cadastral['DATA_CADASTRO'])

# Criando flag binária para pessoa física (1 = PF, 0 = PJ)
# Original: 'X' para PF e NaN para PJ
cadastral['FLAG_PF'] = cadastral['FLAG_PF'].map({'X': 1})
cadastral['FLAG_PF'] = cadastral['FLAG_PF'].fillna(0).astype(int)

In [24]:
# Testando hipótese: nulos em SEGMENTO_INDUSTRIAL e PORTE coincidem com clientes PF?
# Separando os clientes em dois grupos: PF e PJ
clientes_pf = cadastral[cadastral['FLAG_PF'] == 1]
clientes_pj = cadastral[cadastral['FLAG_PF'] == 0]

# Calculando percentual de nulos em SEGMENTO_INDUSTRIAL para cada grupo
pct_nulo_segmento_pf = clientes_pf['SEGMENTO_INDUSTRIAL'].isna().sum() / len(clientes_pf)
pct_nulo_segmento_pj = clientes_pj['SEGMENTO_INDUSTRIAL'].isna().sum() / len(clientes_pj)

# Calculando percentual de nulos em PORTE para cada grupo
pct_nulo_porte_pf = clientes_pf['PORTE'].isna().sum() / len(clientes_pf)
pct_nulo_porte_pj = clientes_pj['PORTE'].isna().sum() / len(clientes_pj)

print(f'SEGMENTO_INDUSTRIAL nulo - PF: {pct_nulo_segmento_pf:.2%}')
print(f'SEGMENTO_INDUSTRIAL nulo - PJ: {pct_nulo_segmento_pj:.2%}')
print(f'PORTE nulo - PF: {pct_nulo_porte_pf:.2%}')
print(f'PORTE nulo - PJ: {pct_nulo_porte_pj:.2%}')

SEGMENTO_INDUSTRIAL nulo - PF: 100.00%
SEGMENTO_INDUSTRIAL nulo - PJ: 1.36%
PORTE nulo - PF: 3.03%
PORTE nulo - PJ: 3.12%


In [25]:
# SEGMENTO_INDUSTRIAL: nulo é estrutural para PF. Tratamos como categoria própria "PESSOA_FISICA".
# Para os poucos nulos remanescentes em PJ, tratamos como "Desconhecido".
cadastral.loc[(cadastral['SEGMENTO_INDUSTRIAL'].isna()) & (cadastral['FLAG_PF'] == 1), 'SEGMENTO_INDUSTRIAL'] = 'PESSOA_FISICA'
cadastral['SEGMENTO_INDUSTRIAL'] = cadastral['SEGMENTO_INDUSTRIAL'].fillna('Desconhecido')

# PORTE: nulos não têm relação com FLAG_PF (distribuição parecida nos dois grupos),
# então tratamos todos como "Desconhecido"
cadastral['PORTE'] = cadastral['PORTE'].fillna('Desconhecido')

# DDD e DOMINIO_EMAIL: nulos tratamos como "Desconhecido"
cadastral['DDD'] = cadastral['DDD'].fillna('Desconhecido')
cadastral['DOMINIO_EMAIL'] = cadastral['DOMINIO_EMAIL'].fillna('Desconhecido')

In [28]:
#Confirmando tratamento de nulos
print('Nulos após tratamento:')
print(cadastral.isna().sum())

Nulos após tratamento:
ID_CLIENTE             0
DATA_CADASTRO          0
DDD                    0
FLAG_PF                0
SEGMENTO_INDUSTRIAL    0
DOMINIO_EMAIL          0
PORTE                  0
CEP_2_DIG              0
dtype: int64


In [27]:
# Verificando duplicatas de linha inteira
print('Linhas duplicadas (todas as colunas):', cadastral.duplicated().sum())

# Verificando duplicatas de ID_CLIENTE (deveria ser único por cliente)
print('IDs duplicados:', cadastral['ID_CLIENTE'].duplicated().sum())


Linhas duplicadas (todas as colunas): 0
IDs duplicados: 0


#### Base Info